In [1]:
# Colab cell: TFLite Micro optimized export (96x96 MobileNetV2-lite + full quantization -> .h)
!pip install -q tensorflow opencv-python-headless

import os, glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
import pathlib

# --- Ajusta estas rutas si hace falta ---
DRIVE_BASE = '/content/drive/MyDrive/Placas'   # tu dataset con subcarpetas ConRelay / SinRelay
OUT_DIR = '/content/relay_tflite_micro'
os.makedirs(OUT_DIR, exist_ok=True)

# Monta Drive si estás en Colab
from google.colab import drive
drive.mount('/content/drive')

# --- Parámetros (pequeños para MCU) ---
IMG_SIZE = (96, 96)
BATCH_SIZE = 16
SEED = 123

# --- (Opcional) función de recorte automática si lo necesitas ---
def maybe_crop_and_save(src_dir, dst_dir):
    # Si ya tienes dataset limpio, puedes omitir esta parte.
    os.makedirs(dst_dir, exist_ok=True)
    for cls in os.listdir(src_dir):
        scls = os.path.join(src_dir, cls)
        if not os.path.isdir(scls): continue
        dcls = os.path.join(dst_dir, cls)
        os.makedirs(dcls, exist_ok=True)
        for f in glob.glob(os.path.join(scls, "*")):
            try:
                img = cv2.imread(f)
                if img is None: continue
                # si quieres, aplica aquí tu crop_pcb_auto() del notebook original
                # por ahora solo copia
                cv2.imwrite(os.path.join(dcls, os.path.basename(f)), img)
            except Exception as e:
                print("skip", f, e)

# Si quieres usar recorte automático, descomenta y personaliza:
# cleaned_dir = '/content/drive/MyDrive/Placas_cleaned'
# maybe_crop_and_save(DRIVE_BASE, cleaned_dir)
# DATA_DIR = cleaned_dir

DATA_DIR = DRIVE_BASE  # usar las carpetas tal cual (ConRelay / SinRelay)

# --- Carga dataset con split correcto ---
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    label_mode='binary',
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    label_mode='binary',
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Store class names before transformations
class_names_from_dataset = train_ds.class_names
print("Classes:", class_names_from_dataset)

# --- Preprocesado: normalización & augmentación (ligera) ---
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

# normalize to 0-1 for training (conversion later handled by quantization)
normalize = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x,y: (normalize(x), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(lambda x,y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(lambda x,y: (normalize(x), y), num_parallel_calls=AUTOTUNE)

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# --- Modelo: MobileNetV2 ligero (sin top) ---
base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    alpha=0.35,           # factor de reducción de tamaño; 0.35 es pequeño
    include_top=False,
    weights='imagenet'    # usar pesos preentrenados ayuda con pocos datos
)

base.trainable = False  # primera fase congelado

inputs = keras.Input(shape=IMG_SIZE + (3,), name='input_image')
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

# --- Entrenamiento cabeza ---
EPOCHS_HEAD = 12
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD)

# --- Fine-tuning: descongelar parte del backbone ---
base.trainable = True
# descongela desde una capa razonable (ajusta si falla por OOM)
fine_tune_at = int(len(base.layers) * 0.6)
for layer in base.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

EPOCHS_FINE = 12
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE)

# --- Evaluar ---
loss, acc = model.evaluate(val_ds)
print("Val accuracy:", acc)

# --- Guardar Keras (opcional) ---
keras_path = os.path.join(OUT_DIR, "modelo_relay_keras.h5")
model.save(keras_path)
print("Saved keras:", keras_path)

# -----------------------
# CONVERSION: TFLite Micro friendly (full int8 / uint8)
# -----------------------
# We'll produce TWO variants: uint8 (easier to feed bytes 0-255) and int8 (slightly smaller sometimes)
# First: prepare representative dataset generator (float images 0..1)
def representative_gen():
    for images, _ in train_ds.take(50):
        # images are normalized to 0..1; convert to float32 as expected
        yield [tf.cast(images, tf.float32)]

# Convert to uint8 full-integer quantized (inference_input_type=uint8)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_gen
# Use UINT8 inputs/outputs for easier Arduino feeding (0..255)
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
# ensure only builtin ops (should be supported on micro)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
tflite_uint8 = converter.convert()
path_uint8 = os.path.join(OUT_DIR, "modelo_relay_uint8.tflite")
open(path_uint8, "wb").write(tflite_uint8)
print("Saved", path_uint8, "size:", os.path.getsize(path_uint8))

# Optionally convert to int8 (if you prefer)
converter2 = tf.lite.TFLiteConverter.from_keras_model(model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
converter2.representative_dataset = representative_gen
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter2.inference_input_type = tf.int8
converter2.inference_output_type = tf.int8
tflite_int8 = converter2.convert()
path_int8 = os.path.join(OUT_DIR, "modelo_relay_int8.tflite")
open(path_int8, "wb").write(tflite_int8)
print("Saved", path_int8, "size:", os.path.getsize(path_int8))

# --- Create C header via xxd for the uint8 variant (recommended)
!xxd -i {path_uint8} > {os.path.join(OUT_DIR, "modelo_relay_uint8.h")}
print("Created header:", os.path.join(OUT_DIR, "modelo_relay_uint8.h"))
print("File sizes (bytes):")
print("  uint8 tflite:", os.path.getsize(path_uint8))
print("  int8  tflite:", os.path.getsize(path_int8))
print("  uint8 header:", os.path.getsize(os.path.join(OUT_DIR, "modelo_relay_uint8.h")))

# --- Download helper (Colab) ---
from google.colab import files
files.download(path_uint8)        # descarga el .tflite (opcional)
files.download(os.path.join(OUT_DIR, "modelo_relay_uint8.h"))  # descarga .h
print(class_names_from_dataset)

Mounted at /content/drive
Found 308 files belonging to 2 classes.
Using 247 files for training.
Found 308 files belonging to 2 classes.
Using 61 files for validation.
Classes: ['Con Relay', 'Sin Relay']
2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_0.35_96             │ (None, 3, 3, 1280)     │       410,208 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 411,489 (1.57 MB)

 Trainable params: 1,281 (5.00 KB)

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['Con Relay', 'Sin Relay']


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

img_size = (96, 96)

def make_ultra_model():
    inputs = keras.Input(shape=(96,96,3))

    x = layers.Rescaling(1./255)(inputs)

    # Bloque 1
    x = layers.Conv2D(8, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    # Bloque 2
    x = layers.Conv2D(16, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    # Bloque 3
    x = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    return keras.Model(inputs, outputs)

model = make_ultra_model()
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(train_ds, validation_data=val_ds, epochs=20)
def representative_dataset():
    for images, _ in train_ds.take(50):
        yield [tf.cast(images, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()
open("relay_ultra_int8.tflite", "wb").write(tflite_model)
print("Done")



Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 48, 48, 8)      │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 8)      │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 32)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,289 (24.57 KB)

 Trainable params: 6,177 (24.13 KB)

 Non-trainable params: 112 (448.00 B)

Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 10s 205ms/step - accuracy: 0.4998 - loss: 0.6973 - val_accuracy: 0.4262 - val_loss: 0.6933
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.6392 - loss: 0.6367 - val_accuracy: 0.4262 - val_loss: 0.6955
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.7410 - loss: 0.5890 - val_accuracy: 0.4262 - val_loss: 0.6984
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.7566 - loss: 0.5614 - val_accuracy: 0.4262 - val_loss: 0.7014
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.7987 - loss: 0.5269 - val_accuracy: 0.4262 - val_loss: 0.7034
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 90ms/step - accuracy: 0.8237 - loss: 0.4903 - val_accuracy: 0.4262 - val_loss: 0.7053
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - accuracy: 0.8645 - loss: 0.4621 - val_accuracy: 0.4262 - val_loss: 0.7086
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.8797 - loss: 0.4302 - val_accuracy: 0.4262

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Done


In [3]:
!xxd -i relay_ultra_int8.tflite > relay_ultra_int8.h
